<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 3

*Надо понять, как получившийся результат соотносится с уже существующими решениями? Нужно взять готовую обученную модель (https://docs.pytorch.org/vision/main/models.html) и посмотреть, какие результаты она дает на imagenette. К сожалению, модели как правило обучают на всём Imagenet1k, то есть они классифицируют на 1000 классов, а в imagenette всего 10. Нужно подумать, что с этим делать.*

The task text suggests using *torchvision* as an interface library for ResNet. We are going to use ResNet50 as the CLIP model we are using (base-vit-patch32) is considered to have same-complexity image encoder

In [ ]:
import transformers
import torch
import requests
import torchvision
from torchvision import transforms
from PIL import Image
from transformers import *
from collections import Counter
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
import torch.nn.functional as F
import gc
import random
import numpy as np
from torchvision.models import resnet50, ResNet50_Weights
import requests

In [ ]:
BATCH_SIZE = 128
imagenet1k_classes = requests.get('https://huggingface.co/api/resolve-cache/datasets/huggingface/label-files/998629752167c60819e6295813eda1c2db248fd4/imagenet-1k-id2label.json?%2Fdatasets%2Fhuggingface%2Flabel-files%2Fresolve%2Fmain%2Fimagenet-1k-id2label.json=&etag=%222c4b82922248728ce97dd737945061aa05e9c309%22').json()

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to('cuda')
model_transforms = ResNet50_Weights.IMAGENET1K_V2.transforms
model.eval()
print('ok')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    model_transforms()
])

imagenette_data = torchvision.datasets.Imagenette('imagenette/', transform=transform, download=True)
data_loader = torch.utils.data.DataLoader(imagenette_data,
                                          batch_size=BATCH_SIZE,
                                          shuffle=True,
                                          num_workers=2,
                                          )

### Sample picking

This cell illustrates that generally, even trained on classifying on 1000 options, Resnet50 does well on task of classifying *Imagenette* pictures. But the accuracy is still lower than needed

In [ ]:
image, cat = data_loader.dataset[random.randint(0,len(data_loader.dataset))]
image = image.to('cuda')
display(torchvision.transforms.functional.to_pil_image(image))

outputs = model.forward(image.unsqueeze(0))

idx = outputs.argmax(dim=1).item()
print()
print(f'Predicted  {imagenet1k_classes[str(idx)]} (class №{idx})')
print(f'Correct    {', '.join(imagenette_data.classes[cat])} (class №{cat})')

### Metrics on raw solution

Let's see what result does bare ResNet50 achieve

In [ ]:
correct_classifications = 0
all_classifications = 0
predictions = []
truth = []

for batch, true_cat in tqdm(data_loader):
  with torch.inference_mode():
    outputs = model.forward(batch.to('cuda'))
    pred_cat = outputs.argmax(dim=1)
    predictions += pred_cat.tolist()
    truth += true_cat.tolist()

In [ ]:
imagenet1k_imagenette_map = {}

for i, classname_tup in enumerate(imagenette_data.classes):
  classname = ', '.join(classname_tup)
  classnames = [c for c in imagenet1k_classes.values()]
  i1k_index = classnames.index(classname)
  imagenet1k_imagenette_map[i1k_index] = i

for i, p in enumerate(predictions):
  idx = imagenet1k_imagenette_map.get(p)
  if idx is not None:
    predictions[i] = idx


In [ ]:
freqs = Counter([11 if (i > 10) else i for i in predictions])
plt.title('Imagenette class sizes')
plt.bar(freqs.keys(), freqs.values())

In [ ]:
truncated_predictions = [11 if (i > 10) else i for i in predictions]

print(f'Accuracy    {metrics.accuracy_score(truth, truncated_predictions)}')
print(f'Precision   {metrics.precision_score(truth, truncated_predictions, average='macro')}')
print(f'Recall      {metrics.recall_score(truth, truncated_predictions, average='macro')}')
print(f'F1          {metrics.f1_score(truth, truncated_predictions, average='macro')}')